# Notebook 5 — Blood-Brain Barrier Permeability (BBBP)

**Project:** Equivariant Graph Neural Networks with Chirality Encoding for Structure-Sensitive Molecular Property Prediction

**Goal of this notebook:** single-task binary classification of whether a molecule crosses the
blood-brain barrier (BBB), using the 3D + stereo-aware encoder and structural fingerprints from
Notebooks 3 and 4. BBBP is a complementary **pharmacokinetic (ADMET)** objective that connects
the framework to central-nervous-system drug design.

**What's different from Notebook 4:**
- **Scaffold split** (standard for BBBP) instead of a random split. Random splits put
  near-identical analogues in train and test and inflate AUC; scaffold splits are the honest
  protocol.
- **Proper retrained ablations** rather than an inference-time probe. We train four variants
  from the same seed and compare them on the same test set:
  1. `fp_only` — Morgan fingerprints only (classical baseline)
  2. `gnn_no_stereo` — 3D EGNN with stereo features masked out
  3. `gnn_stereo` — 3D EGNN with stereo features
  4. `hybrid` — stereo-aware EGNN + fingerprints (the proposed model)

**Honest expectation:** BBBP is small (~2,000 molecules) and its labels depend mostly on
physicochemical properties (polarity, H-bond donors, size) rather than chirality. Do **not**
assume stereo features will raise AUC here — report whatever the ablation shows. With a
dataset this small, differences of ~0.01–0.02 AUC are within seed-to-seed noise, which is why
this notebook can repeat each variant over several seeds.


In [ ]:
!pip install -q torch_geometric
!pip install -q rdkit


In [ ]:
import os
import json
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader as TorchDataLoader
from torch_geometric.datasets import MoleculeNet
from torch_geometric.data import Data, Batch
from torch_geometric.nn import global_mean_pool
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, roc_curve
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

SEED = 42


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)


## 1. Load BBBP via MoleculeNet

Each molecule has one binary label: `1` = penetrates the BBB, `0` = does not. The dataset is
imbalanced (roughly three quarters positive), so we report **ROC-AUC** as the main metric and
also accuracy and F1 for completeness.


In [ ]:
bbbp = MoleculeNet(root='./data/BBBP', name='BBBP')
print(f"Total BBBP molecules: {len(bbbp)}")
print("Sample:", bbbp[0])

labels = torch.cat([bbbp[i].y.view(-1) for i in range(len(bbbp))])
print(f"Positive (penetrates BBB): {int((labels == 1).sum())} "
      f"({100 * (labels == 1).float().mean():.1f}%)")
print(f"Negative: {int((labels == 0).sum())}")


## 2. Stereo-aware graph builder

Same feature scheme as Notebook 4, with two robustness changes for BBBP's messier chemistry:

- **Largest fragment only** — BBBP contains salts/counter-ions (e.g. `.Cl`); we keep the
  largest fragment so counter-ions don't pollute the graph.
- **An `other` atom bucket** — BBBP has rare elements (B, Na, Si, Se, ...). Rather than
  discarding those molecules, they map to a shared `other` one-hot slot.

Node features: atom one-hot (11) + chirality one-hot (3) + signed volume (1) = **15**.
Edge features: bond type (4) + E/Z (3) = **7**.


In [ ]:
ATOM_TYPES = ['H', 'C', 'N', 'O', 'F', 'Cl', 'Br', 'S', 'P', 'I', 'other']
ATOM_TO_IDX = {a: i for i, a in enumerate(ATOM_TYPES)}
N_ATOM = len(ATOM_TYPES)
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE,
              Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]
N_BOND = len(BOND_TYPES)
CHIRALITY_TYPES = ['none', 'R', 'S']
CHIRALITY_TO_IDX = {c: i for i, c in enumerate(CHIRALITY_TYPES)}
STEREO_BOND_TYPES = ['none', 'E', 'Z']
STEREO_BOND_TO_IDX = {c: i for i, c in enumerate(STEREO_BOND_TYPES)}

NODE_DIM = N_ATOM + len(CHIRALITY_TYPES) + 1   # 11 + 3 + 1 = 15
EDGE_DIM = N_BOND + len(STEREO_BOND_TYPES)     # 4 + 3 = 7
print(f"Node dim: {NODE_DIM}, Edge dim: {EDGE_DIM}")


def largest_fragment(mol):
    frags = Chem.GetMolFrags(mol, asMols=True)
    return max(frags, key=lambda m: m.GetNumHeavyAtoms())


def compute_signed_volume(mol, atom_idx, conf):
    atom = mol.GetAtomWithIdx(atom_idx)
    neighbor_idxs = sorted(n.GetIdx() for n in atom.GetNeighbors())
    if len(neighbor_idxs) < 3:
        return 0.0
    p0 = conf.GetAtomPosition(atom_idx)
    p0 = np.array([p0.x, p0.y, p0.z])
    pts = []
    for n_idx in neighbor_idxs[:3]:
        p = conf.GetAtomPosition(n_idx)
        pts.append(np.array([p.x, p.y, p.z]) - p0)
    return float(np.dot(pts[0], np.cross(pts[1], pts[2])))


def embed_one_conformer(mol_noH):
    """ETKDGv3 embedding of a few candidates, MMFF94-optimize, return the
    lowest-energy one. Falls back to random-coordinate embedding for hard
    molecules, and to an unoptimized conformer if MMFF has no parameters."""
    mol = Chem.AddHs(mol_noH)
    params = AllChem.ETKDGv3()
    params.randomSeed = SEED
    cids = list(AllChem.EmbedMultipleConfs(mol, numConfs=5, params=params))
    if len(cids) == 0:
        params.useRandomCoords = True
        cids = list(AllChem.EmbedMultipleConfs(mol, numConfs=5, params=params))
    if len(cids) == 0:
        return None, None

    energies = []
    props = AllChem.MMFFGetMoleculeProperties(mol)
    if props is not None:
        for cid in cids:
            try:
                ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
                ff.Minimize(maxIts=500)
                energies.append((cid, ff.CalcEnergy()))
            except Exception:
                pass
    if len(energies) == 0:
        return mol, cids[0]
    energies.sort(key=lambda t: t[1])
    return mol, energies[0][0]


def mol_to_data_stereo(mol, conf_id, y_value):
    Chem.AssignStereochemistryFrom3D(mol, confId=conf_id)
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
    conf = mol.GetConformer(conf_id)

    atoms = list(mol.GetAtoms())
    x = torch.zeros(len(atoms), NODE_DIM)
    for atom in atoms:
        i = atom.GetIdx()
        x[i, ATOM_TO_IDX.get(atom.GetSymbol(), ATOM_TO_IDX['other'])] = 1.0
        cip = atom.GetProp('_CIPCode') if atom.HasProp('_CIPCode') else None
        x[i, N_ATOM + CHIRALITY_TO_IDX.get(cip, 0)] = 1.0
        if cip is not None:
            x[i, -1] = compute_signed_volume(mol, i, conf)

    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = bond.GetBondType()
        bt_onehot = [1.0 if bt == b else 0.0 for b in BOND_TYPES]
        st = bond.GetStereo()
        if st == Chem.BondStereo.STEREOE:
            s_label = 'E'
        elif st == Chem.BondStereo.STEREOZ:
            s_label = 'Z'
        else:
            s_label = 'none'
        s_onehot = [0.0, 0.0, 0.0]
        s_onehot[STEREO_BOND_TO_IDX[s_label]] = 1.0
        full = bt_onehot + s_onehot
        edge_index += [[i, j], [j, i]]
        edge_attr += [full, full]

    if len(edge_index) == 0:
        return None

    return Data(
        x=x,
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float),
        pos=torch.tensor(conf.GetPositions(), dtype=torch.float),
        y=torch.tensor([y_value], dtype=torch.float),
    )


FP_RADIUS, FP_NBITS = 2, 1024


def compute_morgan_fp(mol_noH):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol_noH, radius=FP_RADIUS, nBits=FP_NBITS)
    return np.array(fp, dtype=np.float32)


## 3. Build (and cache) the dataset

~2,000 molecules, so this is much faster than Tox21. Each sample stores the 3D graph, the
fingerprint, and the cleaned SMILES (needed for the scaffold split).


In [ ]:
CACHE_PATH = 'bbbp_graphs_cache.pkl'


def build_bbbp_dataset(pyg_ds, cache_path):
    if os.path.exists(cache_path):
        print(f"Loading cached dataset from {cache_path}")
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    samples, skipped = [], 0
    for i in tqdm(range(len(pyg_ds)), desc='Building BBBP 3D graphs'):
        raw = pyg_ds[i]
        smiles = getattr(raw, 'smiles', None)
        if smiles is None:
            skipped += 1
            continue
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            skipped += 1
            continue
        mol = largest_fragment(mol)
        clean_smiles = Chem.MolToSmiles(mol)

        mol3d, cid = embed_one_conformer(mol)
        if mol3d is None:
            skipped += 1
            continue
        data = mol_to_data_stereo(mol3d, cid, float(raw.y.view(-1)[0]))
        if data is None:
            skipped += 1
            continue
        samples.append({'data': data, 'fp': compute_morgan_fp(mol), 'smiles': clean_smiles})

    print(f"Built {len(samples)} molecules, skipped {skipped}.")
    with open(cache_path, 'wb') as f:
        pickle.dump(samples, f)
    return samples


all_samples = build_bbbp_dataset(bbbp, CACHE_PATH)


## 4. Scaffold split (80 / 10 / 10)

Molecules are grouped by their **Bemis-Murcko scaffold** (the ring-system core). Whole scaffold
groups go to one split only, with the largest groups assigned to train first. Test molecules
therefore have scaffolds the model never saw in training — a much harder, more realistic test of
generalisation than a random split.


In [ ]:
def scaffold_split(smiles_list, frac_train=0.8, frac_val=0.1):
    groups = {}
    for i, smi in enumerate(smiles_list):
        try:
            sc = MurckoScaffold.MurckoScaffoldSmiles(smiles=smi, includeChirality=False)
        except Exception:
            sc = smi
        groups.setdefault(sc, []).append(i)

    group_list = sorted(groups.values(), key=lambda g: (len(g), g[0]), reverse=True)
    n = len(smiles_list)
    n_train, n_val = frac_train * n, frac_val * n
    train, val, test = [], [], []
    for g in group_list:
        if len(train) + len(g) <= n_train:
            train += g
        elif len(val) + len(g) <= n_val:
            val += g
        else:
            test += g
    return train, val, test


smiles_all = [s['smiles'] for s in all_samples]
train_ids, val_ids, test_ids = scaffold_split(smiles_all)
print(f"Scaffold split -> Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

for name, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    ys = np.array([all_samples[i]['data'].y.item() for i in ids])
    print(f"  {name}: {100 * ys.mean():.1f}% positive")

torch.save({'train': train_ids, 'val': val_ids, 'test': test_ids}, 'bbbp_scaffold_split.pt')


In [ ]:
class BBBPDataset(torch.utils.data.Dataset):
    def __init__(self, samples, ids):
        self.items = [samples[i] for i in ids]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        return it['data'], torch.tensor(it['fp'], dtype=torch.float)


def collate_bbbp(batch):
    graphs, fps = zip(*batch)
    return Batch.from_data_list(list(graphs)), torch.stack(fps)


train_set = BBBPDataset(all_samples, train_ids)
val_set = BBBPDataset(all_samples, val_ids)
test_set = BBBPDataset(all_samples, test_ids)

train_loader = TorchDataLoader(train_set, batch_size=32, shuffle=True, collate_fn=collate_bbbp)
val_loader = TorchDataLoader(val_set, batch_size=64, shuffle=False, collate_fn=collate_bbbp)
test_loader = TorchDataLoader(test_set, batch_size=64, shuffle=False, collate_fn=collate_bbbp)


## 5. Model with switchable branches

One class covers all four ablation variants:

- `use_gnn` / `use_fp` turn the EGNN branch and the fingerprint branch on or off.
- `use_stereo=False` multiplies the chirality, signed-volume and E/Z feature columns by zero
  inside `forward`, so the *same* graphs can be used for the stereo-blind variant (a clean
  comparison — identical molecules, conformers and split).


In [ ]:
class EGNNLayer(nn.Module):
    def __init__(self, hidden_dim, edge_feat_dim):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1 + edge_feat_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU())
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, 1))
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, hidden_dim))

    def forward(self, h, pos, edge_index, edge_attr):
        row, col = edge_index
        rel_pos = pos[row] - pos[col]
        dist_sq = (rel_pos ** 2).sum(dim=-1, keepdim=True)
        m_ij = self.edge_mlp(torch.cat([h[row], h[col], dist_sq, edge_attr], dim=-1))

        coord_weight = self.coord_mlp(m_ij)
        agg_coord = torch.zeros_like(pos)
        agg_coord.index_add_(0, row, rel_pos * coord_weight)
        deg = torch.zeros(pos.size(0), 1, device=pos.device)
        deg.index_add_(0, row, torch.ones_like(coord_weight))
        pos_new = pos + agg_coord / deg.clamp(min=1.0)

        agg_m = torch.zeros_like(h)
        agg_m.index_add_(0, row, m_ij)
        h_new = h + self.node_mlp(torch.cat([h, agg_m], dim=-1))
        return h_new, pos_new


class EGNNEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=64, num_layers=4):
        super().__init__()
        self.embed = nn.Linear(node_in_dim, hidden_dim)
        self.layers = nn.ModuleList(
            [EGNNLayer(hidden_dim, edge_in_dim) for _ in range(num_layers)])

    def forward(self, x, pos, edge_index, edge_attr, batch):
        h = self.embed(x)
        for layer in self.layers:
            h, pos = layer(h, pos, edge_index, edge_attr)
        return global_mean_pool(h, batch)


class BBBPModel(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim=64, num_layers=4,
                 use_gnn=True, use_fp=True, use_stereo=True):
        super().__init__()
        assert use_gnn or use_fp, "at least one branch is required"
        self.use_gnn, self.use_fp, self.use_stereo = use_gnn, use_fp, use_stereo

        if use_gnn:
            self.encoder = EGNNEncoder(node_in_dim, edge_in_dim, hidden_dim, num_layers)
        if use_fp:
            self.fp_branch = nn.Sequential(
                nn.Linear(FP_NBITS, 256), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(256, hidden_dim), nn.ReLU())

        # Column masks that zero out stereo features for the stereo-blind variant.
        node_mask = torch.ones(node_in_dim)
        node_mask[N_ATOM:] = 0.0          # chirality one-hot + signed volume
        edge_mask = torch.ones(edge_in_dim)
        edge_mask[N_BOND:] = 0.0          # E/Z one-hot
        self.register_buffer('node_mask', node_mask)
        self.register_buffer('edge_mask', edge_mask)

        in_dim = hidden_dim * (int(use_gnn) + int(use_fp))
        self.classifier = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, 1))

    def forward(self, batch, fp):
        parts = []
        if self.use_gnn:
            x, edge_attr = batch.x, batch.edge_attr
            if not self.use_stereo:
                x = x * self.node_mask
                edge_attr = edge_attr * self.edge_mask
            parts.append(self.encoder(x, batch.pos, batch.edge_index, edge_attr, batch.batch))
        if self.use_fp:
            parts.append(self.fp_branch(fp))
        return self.classifier(torch.cat(parts, dim=-1)).view(-1)   # raw logits (B,)


## 6. Training and evaluation utilities

Model selection uses **validation ROC-AUC**; the best-validation checkpoint is then evaluated
once on the scaffold-split test set.


In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total, n = 0.0, 0
    for batch, fp in loader:
        batch, fp = batch.to(device), fp.to(device)
        optimizer.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(batch, fp), batch.y.view(-1))
        loss.backward()
        optimizer.step()
        total += loss.item() * batch.num_graphs
        n += batch.num_graphs
    return total / max(n, 1)


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    probs, ys = [], []
    for batch, fp in loader:
        batch, fp = batch.to(device), fp.to(device)
        probs.append(torch.sigmoid(model(batch, fp)).cpu())
        ys.append(batch.y.view(-1).cpu())
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()


def compute_metrics(probs, ys):
    preds = (probs >= 0.5).astype(int)
    return {
        'roc_auc': float(roc_auc_score(ys, probs)),
        'accuracy': float(accuracy_score(ys, preds)),
        'f1': float(f1_score(ys, preds)),
    }


def train_variant(name, seed, use_gnn, use_fp, use_stereo, n_epochs=50, verbose=False):
    set_seed(seed)
    model = BBBPModel(NODE_DIM, EDGE_DIM, use_gnn=use_gnn, use_fp=use_fp,
                      use_stereo=use_stereo).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',
                                                           factor=0.5, patience=5)
    ckpt = f'bbbp_{name}_seed{seed}.pt'
    best_val, history = -1.0, {'train_loss': [], 'val_auc': []}

    for epoch in range(1, n_epochs + 1):
        loss = train_one_epoch(model, train_loader, optimizer, device)
        vp, vy = predict(model, val_loader, device)
        val_auc = roc_auc_score(vy, vp)
        scheduler.step(val_auc)
        history['train_loss'].append(loss)
        history['val_auc'].append(float(val_auc))
        if val_auc > best_val:
            best_val = val_auc
            torch.save(model.state_dict(), ckpt)
        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f"  [{name} s{seed}] epoch {epoch:03d} | loss {loss:.4f} | val AUC {val_auc:.4f}")

    model.load_state_dict(torch.load(ckpt, map_location=device))
    tp, ty = predict(model, test_loader, device)
    metrics = compute_metrics(tp, ty)
    metrics['best_val_auc'] = float(best_val)
    return metrics, history, (tp, ty)


## 7. Run the four variants over multiple seeds

Each variant is trained with 3 seeds (same seeds across variants), so we can report
**mean ± std** instead of a single lucky/unlucky run. On a free Colab GPU this takes a few
minutes in total; reduce `SEEDS` to `[42]` for a quick first pass.


In [ ]:
VARIANTS = {
    'fp_only':       dict(use_gnn=False, use_fp=True,  use_stereo=True),
    'gnn_no_stereo': dict(use_gnn=True,  use_fp=False, use_stereo=False),
    'gnn_stereo':    dict(use_gnn=True,  use_fp=False, use_stereo=True),
    'hybrid':        dict(use_gnn=True,  use_fp=True,  use_stereo=True),
}
SEEDS = [42, 43, 44]
N_EPOCHS = 50

all_runs = {name: [] for name in VARIANTS}
histories = {}
test_preds = {}

for name, cfg in VARIANTS.items():
    for seed in SEEDS:
        print(f"Training {name} (seed {seed}) ...")
        metrics, history, preds = train_variant(name, seed, n_epochs=N_EPOCHS, **cfg)
        all_runs[name].append(metrics)
        histories[(name, seed)] = history
        test_preds[(name, seed)] = preds
        print(f"   test AUC {metrics['roc_auc']:.4f} | acc {metrics['accuracy']:.4f} "
              f"| F1 {metrics['f1']:.4f} | best val AUC {metrics['best_val_auc']:.4f}")


## 8. Results table (mean ± std over seeds)


In [ ]:
summary = {}
print(f"{'Variant':16s} {'Test ROC-AUC':>18s} {'Accuracy':>16s} {'F1':>16s}")
print("-" * 70)
for name, runs in all_runs.items():
    row = {}
    for m in ['roc_auc', 'accuracy', 'f1']:
        vals = np.array([r[m] for r in runs])
        row[m] = {'mean': float(vals.mean()), 'std': float(vals.std()), 'values': vals.tolist()}
    summary[name] = row
    print(f"{name:16s} "
          f"{row['roc_auc']['mean']:.4f} ± {row['roc_auc']['std']:.4f}   "
          f"{row['accuracy']['mean']:.4f} ± {row['accuracy']['std']:.4f}   "
          f"{row['f1']['mean']:.4f} ± {row['f1']['std']:.4f}")

print()
print("How to read this: if two variants' mean AUCs differ by less than about one std, "
      "treat them as indistinguishable on this dataset size.")


## 9. ROC curves and training curves


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

colors = {'fp_only': 'tab:gray', 'gnn_no_stereo': 'tab:orange',
          'gnn_stereo': 'tab:green', 'hybrid': 'tab:blue'}
for name in VARIANTS:
    probs, ys = test_preds[(name, SEEDS[0])]
    fpr, tpr, _ = roc_curve(ys, probs)
    auc = summary[name]['roc_auc']['mean']
    axes[0].plot(fpr, tpr, color=colors[name], label=f"{name} (mean AUC {auc:.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate')
axes[0].set_title(f'BBBP test ROC (seed {SEEDS[0]}, scaffold split)')
axes[0].legend(fontsize=8)

for name in VARIANTS:
    axes[1].plot(histories[(name, SEEDS[0])]['val_auc'], color=colors[name], label=name)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation ROC-AUC')
axes[1].set_title('Validation AUC during training')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## 10. Save results for the final comparison notebook


In [ ]:
results_bbbp = {
    'dataset': 'BBBP',
    'split': 'scaffold (80/10/10)',
    'num_train': len(train_set),
    'num_val': len(val_set),
    'num_test': len(test_set),
    'seeds': SEEDS,
    'epochs': N_EPOCHS,
    'variants': summary,
    'proposed_model': 'hybrid',
}

with open('results_bbbp.json', 'w') as f:
    json.dump(results_bbbp, f, indent=2)

print(json.dumps({k: {m: round(v['mean'], 4) for m, v in row.items()}
                  for k, row in summary.items()}, indent=2))


## Done — files to keep for Notebook 6 (final comparison)

| File | Contents |
|---|---|
| `results_bbbp.json` | Mean ± std test metrics for all four variants |
| `bbbp_scaffold_split.pt` | The scaffold split indices |
| `bbbp_graphs_cache.pkl` | Cached 3D graphs (optional; regenerable) |

**How to discuss the results in the viva:**
- **Why a scaffold split:** it prevents near-duplicate analogues leaking from train to test, so
  the AUC reflects generalisation to new chemotypes. Expect lower numbers than random-split
  papers report — that is the honest number.
- **If `gnn_stereo` ≈ `gnn_no_stereo`:** that is a legitimate finding, not a failure. BBB
  permeability is driven mainly by polarity, hydrogen-bonding and molecular size; chirality
  rarely changes passive permeability. Stereo features matter more for target-binding endpoints.
- **If `fp_only` is competitive with the GNN variants:** also expected on ~2,000 molecules,
  where fingerprints are a strong, low-variance baseline. The hybrid is justified only if it
  matches or beats it — report whichever is true.
- **Variance:** quote mean ± std, and avoid claiming a winner when differences fall within one
  standard deviation.

**Next up (Notebook 6):** load `results_*.json` from Notebooks 1–5 and build the final
comparison — 2D GNN → 3D GNN → 3D + Stereo → multi-task Tox21 / BBBP — with tables and plots.
